# InduSense — Sprint 1 : cartes d'identité des fichiers

Pour chaque fichier de `datas/` (CSV et script SQL), ce notebook produit une carte d'identité Markdown dans
`artifacts/ingestion/identities/<AAAAMMJJ>/<nom_fichier>_<HHMMSS>.md` : source, taille, horaires de traitement,
charset, dimensions, types de colonnes inférés et statistiques des colonnes numériques et dates.

In [1]:
import csv
from datetime import datetime
from decimal import Decimal
from pathlib import Path

import pandas as pd
import sqlglot
from IPython.display import Markdown, display
from sqlglot import exp

DATA_DIR = Path("datas")
ARTIFACTS_DIR = Path("artifacts/ingestion/identities")
CSV_FILES = [DATA_DIR / "releves_incidents.csv", DATA_DIR / "telemetry.csv"]
SQL_FILE = DATA_DIR / "machine.sql.txt"
SQL_DIALECT = "postgres"

## 1. Lecture du fichier : taille, charset et séparateur

- **Charset** : on essaie de décoder le fichier en ASCII, puis en UTF-8 (BOM éventuel ignoré) ;
  à défaut il est lu en ISO-8859-1 (Latin-1), qui accepte n'importe quel octet.
- **Séparateur** : il n'est pas toujours une virgule, `csv.Sniffer` le détecte parmi `,` `;` `\t` `|`.

In [2]:
def format_filesize(size: float) -> str:
    """Formate une taille en octets : 9999.99 [octets | Ko | Mo | Go]."""
    for unit in ("octets", "Ko", "Mo"):
        if size < 1024:
            return f"{size:.2f} {unit}"
        size /= 1024
    return f"{size:.2f} Go"


def detect_charset(path: Path) -> tuple[str, str]:
    """Retourne (libellé du charset, encodage Python pour lire le fichier)."""
    raw = path.read_bytes()
    for label, encoding in (("ASCII", "ascii"), ("UTF-8", "utf-8-sig")):
        try:
            raw.decode(encoding)
            return label, encoding
        except UnicodeDecodeError:
            pass
    return "ISO-8859-1 (Latin-1)", "latin-1"


def detect_separator(path: Path, encoding: str) -> str:
    """Détecte le séparateur du CSV sur un échantillon du début du fichier."""
    with path.open(encoding=encoding, newline="") as f:
        sample = f.read(64 * 1024)
    try:
        return csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
    except csv.Error:
        return ","

## 2. Inférence du type des colonnes

| Type | Règle |
| --- | --- |
| `IDENTITY` | nom `id` / `*_id` / `*_code`, ou valeurs de la forme `MACH-01`, `INC-000001`, `OP1002` (au moins 2 chiffres) |
| `BOOLEAN` | valeurs uniquement `0` / `1` / `true` / `false` (casse ignorée, mélange possible) |
| `INTEGER` / `FLOAT` | colonne numérique ; décimales avec point (`12.5`) ou virgule (`12,5`), éventuellement mélangées |
| `TIME` | heures `HH:MM` ou `HH:MM:SS` (`5:42` accepté) |
| `DATE` / `DATETIME` | ISO 8601, `JJ/MM/AAAA`, `JJ-MM-AAAA` ou `AAAA/MM/JJ`, avec ou sans heure ; `DATETIME` si les valeurs portent une heure |
| `CATEGORY` | texte avec au plus 20 valeurs distinctes |
| `TEXT` | autre texte |

Les valeurs nulles (vide, `null`, `NULL`...) sont ignorées : pandas et sqlglot les lisent déjà comme manquantes.

Pour les heures et dates, chaque format candidat est testé et on garde celui qui lit le plus de valeurs.
La colonne est retenue si **au moins 95 %** des valeurs sont lisibles : une faute de saisie isolée ne la fait
plus basculer en `TEXT`. Les valeurs illisibles sont exclues des statistiques et comptées dans la carte,
ex. `DATE (2 invalid values)`. Les dates avec fuseau horaire sont ramenées en UTC.

In [3]:
IDENTITY_PATTERN = r"[A-Za-z]+-?\d{2,}"
BOOLEAN_VALUES = {"0", "1", "0.0", "1.0", "true", "false"}
MAX_CATEGORIES = 20
TEMPORAL_THRESHOLD = 0.95  # part minimale de valeurs lisibles pour une colonne heure / date
TEMPORAL_FORMATS = [  # testés dans l'ordre ; DATE devient DATETIME si les valeurs portent une heure
    ("TIME", "%H:%M"), ("TIME", "%H:%M:%S"),
    ("DATE", "ISO8601"),
    ("DATE", "%d/%m/%Y"), ("DATE", "%d/%m/%Y %H:%M"), ("DATE", "%d/%m/%Y %H:%M:%S"),
    ("DATE", "%d-%m-%Y"), ("DATE", "%d-%m-%Y %H:%M"), ("DATE", "%d-%m-%Y %H:%M:%S"),
    ("DATE", "%Y/%m/%d"), ("DATE", "%Y/%m/%d %H:%M"), ("DATE", "%Y/%m/%d %H:%M:%S"),
]
NUMERIC_TYPES = {"INTEGER", "FLOAT"}
TEMPORAL_TYPES = {"DATE", "DATETIME", "TIME"}
STATS_TYPES = NUMERIC_TYPES | TEMPORAL_TYPES


def parse_temporal(series: pd.Series) -> tuple[str, pd.Series] | None:
    """(TIME | DATE | DATETIME, valeurs converties avec NaT si illisibles) selon le format qui lit le plus
    de valeurs, ou None si moins de TEMPORAL_THRESHOLD des valeurs sont lisibles."""
    text = series.dropna().astype(str).str.strip()
    best_ratio, best = 0.0, None
    for col_type, fmt in TEMPORAL_FORMATS:
        parsed = pd.to_datetime(text, format=fmt, errors="coerce", utc=True)
        ratio = parsed.notna().mean()
        if ratio > best_ratio:
            best_ratio, best = ratio, (col_type, parsed)
        if ratio == 1:
            break
    if best_ratio < TEMPORAL_THRESHOLD:
        return None
    col_type, parsed = best
    valid = parsed.dropna()
    has_time = (valid != valid.dt.normalize()).any() or text[parsed.notna()].str.len().max() > 10
    return ("DATETIME" if col_type == "DATE" and has_time else col_type), parsed


def infer_column_type(series: pd.Series) -> str:
    """Infère le type métier d'une colonne à partir de son nom et de ses valeurs."""
    name, values = str(series.name).lower(), series.dropna()
    if values.empty:
        return "EMPTY"
    if name == "id" or name.endswith(("_id", "_code")):
        return "IDENTITY"
    if {str(v).strip().lower() for v in values.unique()} <= BOOLEAN_VALUES:
        return "BOOLEAN"
    if pd.api.types.is_numeric_dtype(values):
        return "INTEGER" if pd.api.types.is_integer_dtype(values) else "FLOAT"

    text = values.astype(str)
    if text.str.fullmatch(IDENTITY_PATTERN).all():
        return "IDENTITY"
    if temporal := parse_temporal(values):
        return temporal[0]
    return "CATEGORY" if text.nunique() <= MAX_CATEGORIES else "TEXT"


def type_label(series: pd.Series, col_type: str) -> str:
    """Type inféré, complété du nombre de valeurs illisibles pour une colonne heure / date."""
    if col_type in TEMPORAL_TYPES and (invalid := parse_temporal(series)[1].isna().sum()):
        return f"{col_type} ({invalid} invalid values)"
    return col_type

## 3. Statistiques des colonnes numériques et dates

Min et max pour toutes ces colonnes ; mean et median uniquement pour les numériques.

In [4]:
def column_stats(series: pd.Series, col_type: str) -> list[str]:
    """Retourne [min, max, mean, median] formatés ; mean et median valent "-" hors numériques."""
    values = series.dropna()
    if col_type in NUMERIC_TYPES:
        fmt = "{}" if col_type == "INTEGER" else "{:.3f}"
        return [fmt.format(values.min()), fmt.format(values.max()), f"{values.mean():.3f}", f"{values.median():.3f}"]
    dates = parse_temporal(values)[1].dropna()  # valeurs illisibles exclues
    if col_type == "TIME":
        date_format = "%H:%M:%S" if dates.dt.second.any() else "%H:%M"
    else:
        date_format = "%Y-%m-%d" if col_type == "DATE" else "%Y-%m-%d %H:%M:%S"
    return [dates.min().strftime(date_format), dates.max().strftime(date_format), "-", "-"]

## 4. Construction de la carte d'identité

`write_card` produit l'en-tête commun à tous les fichiers (source, taille, horaires, charset) puis délègue
le corps de la carte à une fonction `describe` propre au format : `describe_csv` ici, `describe_sql` en section 5.

In [5]:
def md_table(headers: list[str], rows: list[list]) -> list[str]:
    """Lignes d'un tableau Markdown."""
    return [
        f"| {' | '.join(headers)} |",
        "|" + " --- |" * len(headers),
        *(f"| {' | '.join(map(str, row))} |" for row in rows),
    ]


def describe_frame(df: pd.DataFrame, sql_types: dict | None = None, level: str = "##") -> list[str]:
    """Dimensions, types inférés (et types SQL déclarés) et statistiques d'un DataFrame."""
    types = {col: infer_column_type(df[col]) for col in df.columns}
    labels = {col: type_label(df[col], t) for col, t in types.items()}
    if sql_types:  # colonnes déclarées, y compris celles absentes des données (valeur par défaut)
        columns = list(sql_types) + [col for col in types if col not in sql_types]
        column_list = md_table(
            ["Colname", "SQL Type", "Inferred Type"],
            [[col, sql_types.get(col, "-"), labels.get(col, "-")] for col in columns],
        )
    else:
        columns = list(types)
        column_list = md_table(["Colname", "Inferred Type"], labels.items())
    stats = [[col, *column_stats(df[col], t)] for col, t in types.items() if t in STATS_TYPES]
    return [
        f"- Rows Number: {len(df)}",
        f"- Cols Number: {len(columns)}",
        "", f"{level} Column list", "", *column_list,
        "", f"{level} Numeric and date columns", "",
        *md_table(["Colname", "Min value", "Max value", "Mean value", "Median value"], stats),
    ]


DECIMAL_PATTERN = r"[+-]?\d+(?:[.,]\d+)?"  # 12 | 12.5 | 12,5 | -3,25


def parse_decimals(df: pd.DataFrame) -> pd.DataFrame:
    """Convertit en nombres les colonnes texte dont toutes les valeurs sont des nombres à point ou à virgule."""
    for col in df.columns:
        text = df[col].dropna().astype(str).str.strip()
        if not pd.api.types.is_numeric_dtype(df[col]) and len(text) and text.str.fullmatch(DECIMAL_PATTERN).all():
            df[col] = pd.to_numeric(text.str.replace(",", ".", regex=False)).reindex(df.index)
    return df


def describe_csv(path: Path, encoding: str) -> list[str]:
    """Corps de la carte d'identité d'un fichier CSV."""
    sep = detect_separator(path, encoding)
    return describe_frame(parse_decimals(pd.read_csv(path, sep=sep, encoding=encoding, skipinitialspace=True)))


def write_card(path: Path, describe) -> Path:
    """Écrit la carte dans ARTIFACTS_DIR/<AAAAMMJJ>/<nom_fichier>_<HHMMSS>.md et retourne son chemin."""
    begin = datetime.now()
    charset, encoding = detect_charset(path)
    body = describe(path, encoding)
    end = datetime.now()
    lines = [
        f"# Carte d'identité : {path.name}",
        "",
        f"- Source: {path.resolve()}",
        f"- Filesize: {format_filesize(path.stat().st_size)}",
        f"- ProcessBeginAt: {begin:%H:%M:%S}",
        f"- ProcessEndAt: {end:%H:%M:%S}",
        f"- Charset: {charset}",
        *body,
    ]
    output = ARTIFACTS_DIR / f"{begin:%Y%m%d}" / f"{path.name.split('.')[0]}_{begin:%H%M%S}.md"
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return output

## 5. Pour aller plus loin : carte d'identité du script SQL

`machine.sql.txt` est un script PostgreSQL : il ne se lit pas comme un tableau. Il faut un parseur SQL
capable d'en extraire deux informations :

- `CREATE TABLE` donne les colonnes et leur **type SQL déclaré**,
- `INSERT INTO ... VALUES` donne les **données**, chargées dans un DataFrame par table.

Chaque table est ensuite décrite comme un CSV, avec la même fonction `describe_frame`. Les colonnes déclarées
mais absentes des `INSERT` (remplies par une valeur par défaut, ex. `created_at`) ont le type inféré `-`.

### Choix de la librairie : sqlglot

| Option | Ce qu'elle fournit | Pourquoi elle n'a pas été retenue |
| --- | --- | --- |
| **[sqlglot](https://github.com/tobymao/sqlglot)** ✅ | Arbre syntaxique complet (AST) : tables, colonnes, types, valeurs déjà typées (`int`, `Decimal`, `None`...) | — |
| [sqlparse](https://github.com/andialbrecht/sqlparse) | Découpage en instructions et en *tokens* (parseur « non validant ») | Pas d'arbre syntaxique : il faudrait reconstruire à la main colonnes, types et lignes de `VALUES` en parcourant les tokens |
| [pglast](https://github.com/lelit/pglast) | Le vrai parseur de PostgreSQL (libpg_query) | Extension C : demande des binaires compilés pour notre version de Python (3.15) ; limitée au seul dialecte PostgreSQL |
| `sqlite3` (bibliothèque standard) | Exécute le script dans une base en mémoire, puis on relit les tables | Ce n'est pas un parseur : il faut que SQLite accepte le script, or celui-ci utilise des syntaxes propres à PostgreSQL (`DEFAULT NOW()`, `TIMESTAMPTZ`...) ; et on perd les types déclarés |

sqlglot coche toutes les cases pour ce besoin :

- **Python pur, sans aucune dépendance** : `uv add sqlglot` n'installe qu'un seul paquet, rien à compiler,
- **comprend le dialecte PostgreSQL** (`read="postgres"`) : `TIMESTAMPTZ`, `NUMERIC(6,2)`, `ON CONFLICT ... DO UPDATE`...,
- **aucune base de données nécessaire** : le script est analysé, jamais exécuté,
- **arbre syntaxique navigable** : `exp.Create`, `exp.Insert`, `exp.ColumnDef`... rendent l'extraction directe,
- projet très utilisé et activement maintenu.

```shell
uv add sqlglot
```

In [6]:
def sql_value(node: exp.Expression):
    """Valeur Python d'un littéral SQL (les NUMERIC, lus en Decimal, deviennent des float)."""
    value = node.to_py()
    return float(value) if isinstance(value, Decimal) else value


def extract_sql_tables(statements: list[exp.Expression]) -> dict[str, tuple[dict, pd.DataFrame]]:
    """Pour chaque table : (types déclarés par CREATE TABLE, données des INSERT ... VALUES)."""
    sql_types, frames = {}, {}
    for statement in statements:
        if isinstance(statement, exp.Create) and statement.kind == "TABLE":
            schema = statement.this
            sql_types[schema.this.name] = {
                col.name: col.args["kind"].sql(SQL_DIALECT)
                for col in schema.expressions
                if isinstance(col, exp.ColumnDef)
            }
        elif isinstance(statement, exp.Insert) and isinstance(statement.expression, exp.Values):
            target = statement.this
            if isinstance(target, exp.Schema):  # INSERT INTO table (col1, col2, ...)
                name, columns = target.this.name, [col.name for col in target.expressions]
            else:  # INSERT INTO table VALUES ... : toutes les colonnes déclarées
                name, columns = target.name, list(sql_types.get(target.name, {}))
            rows = [[sql_value(v) for v in row.expressions] for row in statement.expression.expressions]
            frames.setdefault(name, []).append(pd.DataFrame(rows, columns=columns))
    return {
        name: (sql_types.get(name, {}), pd.concat(frames[name], ignore_index=True) if name in frames else pd.DataFrame())
        for name in dict.fromkeys([*sql_types, *frames])
    }


def describe_sql(path: Path, encoding: str) -> list[str]:
    """Corps de la carte d'identité d'un script SQL : une section par table."""
    statements = sqlglot.parse(path.read_text(encoding=encoding), read=SQL_DIALECT)
    tables = extract_sql_tables(statements)
    lines = [
        f"- SQL Dialect: {SQL_DIALECT}",
        f"- Statements Number: {len(statements)}",
        f"- Tables Number: {len(tables)}",
    ]
    for name, (sql_types, df) in tables.items():
        lines += ["", f"## Table `{name}`", "", *describe_frame(df, sql_types, level="###")]
    return lines

## 6. Génération des cartes d'identité

In [7]:
CARDS = {path: describe_csv for path in CSV_FILES} | {SQL_FILE: describe_sql}

for path, describe in CARDS.items():
    output = write_card(path, describe)
    print(f"Carte générée : {output}")
    display(Markdown(output.read_text(encoding="utf-8")))

Carte générée : artifacts/ingestion/identities/20261120/releves_incidents_154937.md


# Carte d'identité : releves_incidents.csv

- Source: /home/thopuetai/Formation IA/indusense/datas/releves_incidents.csv
- Filesize: 134.64 Ko
- ProcessBeginAt: 15:49:37
- ProcessEndAt: 15:49:37
- Charset: UTF-8
- Rows Number: 1245
- Cols Number: 18

## Column list

| Colname | Inferred Type |
| --- | --- |
| incident_id | IDENTITY |
| date | DATE |
| time | TIME |
| operator_name | CATEGORY |
| machine_id | IDENTITY |
| severity | INTEGER |
| operator_badge | IDENTITY |
| comment | TEXT |
| shift | CATEGORY |
| type_surchauffe | BOOLEAN |
| type_baisse_pression | BOOLEAN |
| type_vibration | BOOLEAN |
| type_bruit_mecanique | BOOLEAN |
| type_surconsommation | BOOLEAN |
| type_blocage_mecanique | BOOLEAN |
| type_alarme_capteur | BOOLEAN |
| type_arret_urgence | BOOLEAN |
| type_defaut_qualite | BOOLEAN |

## Numeric and date columns

| Colname | Min value | Max value | Mean value | Median value |
| --- | --- | --- | --- | --- |
| date | 2025-06-01 | 2026-06-08 | - | - |
| time | 00:06 | 23:55 | - | - |
| severity | 2 | 5 | 2.925 | 3.000 |


Carte générée : artifacts/ingestion/identities/20261120/telemetry_154937.md


# Carte d'identité : telemetry.csv

- Source: /home/thopuetai/Formation IA/indusense/datas/telemetry.csv
- Filesize: 7.94 Mo
- ProcessBeginAt: 15:49:37
- ProcessEndAt: 15:49:38
- Charset: ASCII
- Rows Number: 135626
- Cols Number: 7

## Column list

| Colname | Inferred Type |
| --- | --- |
| machine_id | IDENTITY |
| timestamp | DATETIME |
| temperature_c | FLOAT |
| pressure_bar | FLOAT |
| voltage_mean_v | FLOAT |
| rotation_mean_rpm | FLOAT |
| pieces_produced | INTEGER |

## Numeric and date columns

| Colname | Min value | Max value | Mean value | Median value |
| --- | --- | --- | --- | --- |
| timestamp | 2025-06-01 00:00:00 | 2026-06-08 23:00:00 | - | - |
| temperature_c | 32.000 | 80.000 | 48.183 | 48.055 |
| pressure_bar | 159.982 | 215.814 | 199.770 | 199.866 |
| voltage_mean_v | 220.900 | 242.000 | 227.631 | 227.420 |
| rotation_mean_rpm | 1100.000 | 1900.000 | 1589.205 | 1590.370 |
| pieces_produced | 0 | 114 | 49.533 | 49.000 |


Carte générée : artifacts/ingestion/identities/20261120/machine_154938.md


# Carte d'identité : machine.sql.txt

- Source: /home/thopuetai/Formation IA/indusense/datas/machine.sql.txt
- Filesize: 22.41 Ko
- ProcessBeginAt: 15:49:38
- ProcessEndAt: 15:49:38
- Charset: UTF-8
- SQL Dialect: postgres
- Statements Number: 10
- Tables Number: 2

## Table `machine`

- Rows Number: 15
- Cols Number: 11

### Column list

| Colname | SQL Type | Inferred Type |
| --- | --- | --- |
| machine_code | VARCHAR(16) | IDENTITY |
| commissioning_date | DATE | DATE |
| max_daily_capacity | INT | INTEGER |
| max_hourly_capacity_pieces | INT | INTEGER |
| model | VARCHAR(32) | CATEGORY |
| production_line | VARCHAR(16) | CATEGORY |
| location | VARCHAR(16) | CATEGORY |
| criticality | VARCHAR(8) | CATEGORY |
| is_active | BOOLEAN | - |
| created_at | TIMESTAMPTZ | - |
| updated_at | TIMESTAMPTZ | - |

### Numeric and date columns

| Colname | Min value | Max value | Mean value | Median value |
| --- | --- | --- | --- | --- |
| commissioning_date | 2019-07-23 | 2025-05-25 | - | - |
| max_daily_capacity | 750 | 1428 | 1054.867 | 1027.000 |
| max_hourly_capacity_pieces | 47 | 89 | 65.867 | 64.000 |

## Table `maintenance`

- Rows Number: 115
- Cols Number: 11

### Column list

| Colname | SQL Type | Inferred Type |
| --- | --- | --- |
| maintenance_id | INT | IDENTITY |
| machine_code | VARCHAR(16) | IDENTITY |
| maintenance_at | TIMESTAMPTZ | DATETIME |
| maintenance_type | VARCHAR(16) | CATEGORY |
| action_type | VARCHAR(32) | CATEGORY |
| component | VARCHAR(64) | CATEGORY |
| description | TEXT | TEXT |
| related_incident_id | VARCHAR(16) | IDENTITY |
| duration_hours | DECIMAL(6, 2) | FLOAT |
| created_at | TIMESTAMPTZ | - |
| updated_at | TIMESTAMPTZ | - |

### Numeric and date columns

| Colname | Min value | Max value | Mean value | Median value |
| --- | --- | --- | --- | --- |
| maintenance_at | 2025-06-15 06:00:00 | 2026-05-30 22:00:00 | - | - |
| duration_hours | 1.000 | 8.000 | 2.943 | 2.640 |
